In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

In [2]:
estoque_inicial = 500
media_demanda = 30
desvio_demanda = 50

lead_time = 5
lote_compra = 400

n_dias = 365
n_simulacoes = 1000

In [3]:
def simular_estoque(ROP):

    estoque = estoque_inicial
    pedidos_em_transito = []
    
    historico_estoque = []
    
    for dia in range(n_dias):
        
        # chegada de pedidos
        pedidos_chegaram = [p for p in pedidos_em_transito if p[0] == dia]
        for p in pedidos_chegaram:
            estoque += p[1]
            
        pedidos_em_transito = [p for p in pedidos_em_transito if p[0] > dia]
        
        # demanda do dia
        demanda = max(0, int(np.random.normal(media_demanda, desvio_demanda)))
        estoque -= demanda
        
        # posição de estoque
        posicao_estoque = estoque + sum([p[1] for p in pedidos_em_transito])
        
        # regra de compra
        if posicao_estoque <= ROP:
            chegada = dia + lead_time
            pedidos_em_transito.append((chegada, lote_compra))
        
        historico_estoque.append(estoque)
    
    return historico_estoque

In [4]:
def rodar_simulacoes(ROP):

    resultados = []
    minimos = []
    
    for i in range(n_simulacoes):
        
        estoque = simular_estoque(ROP)
        
        resultados.append(estoque)
        minimos.append(min(estoque))
        
    return np.array(resultados), np.array(minimos)

In [5]:
rops = range(100, 600, 25)

resultados_servico = []

for rop in rops:
    
    simulacoes, minimos = rodar_simulacoes(rop)
    
    faltas = np.sum(minimos <= 0)
    
    nivel_servico = 1 - (faltas / n_simulacoes)
    
    resultados_servico.append((rop, nivel_servico))

df_servico = pd.DataFrame(resultados_servico, columns=["ROP","NivelServico"])

df_servico

,ROP,NivelServico
0,100,0.000
1,125,0.000
2,150,0.000
3,175,0.000
4,200,0.000
5,225,0.000
6,250,0.000
7,275,0.005
8,300,0.031
9,325,0.080


In [6]:
fig = px.line(df_servico,
              x="ROP",
              y="NivelServico",
              title="ROP vs Nível de Serviço")

fig.show()

In [7]:
simulacoes, minimos = rodar_simulacoes(350)

fig = px.line(simulacoes[0],
              title="Evolução do estoque em 365 dias")

fig.show()

In [8]:
fig = px.histogram(minimos,
                   title="Distribuição do estoque mínimo",
                   nbins=40)

fig.show()

In [9]:
desvio_demanda = 150

In [10]:
simulacoes, minimos = rodar_simulacoes(350)

faltas = np.sum(minimos <= 0)

nivel_servico = 1 - (faltas / n_simulacoes)

nivel_servico

np.float64(0.0)

In [11]:
rops = range(300, 1000, 50)

resultados_servico = []

for rop in rops:
    
    simulacoes, minimos = rodar_simulacoes(rop)
    
    faltas = np.sum(minimos <= 0)
    
    nivel_servico = 1 - (faltas / n_simulacoes)
    
    resultados_servico.append((rop, nivel_servico))

df_servico2 = pd.DataFrame(resultados_servico, columns=["ROP","NivelServico"])

df_servico2

,ROP,NivelServico
0,300,0.000
1,350,0.000
2,400,0.000
3,450,0.000
4,500,0.000
5,550,0.000
6,600,0.000
7,650,0.002
8,700,0.005
9,750,0.013
